Colab 5 — Continued Pretraining (Language Adaptation) with Unsloth

In [1]:
import torch, os, sys
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi -L || echo "No GPU detected — enable one via Runtime → Change runtime type → GPU."


Torch: 2.8.0+cu126
CUDA available: True
GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-96a8122a-aece-08c9-2750-0d4a273cfc39)


In [2]:
!pip -q install -U "unsloth>=2025.9.0" "transformers>=4.45.0" \
                 "datasets>=2.20.0" "accelerate>=1.0.0" \
                 "bitsandbytes>=0.43.0" "peft>=0.13.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 19.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [3]:
# Model & training knobs
BASE_MODEL   = "HuggingFaceTB/SmolLM2-135M-Instruct"  # swap to a base/foundation variant if you have it
MAX_SEQ_LEN  = 1024
USE_4BIT     = False       # True for QLoRA (less VRAM, slower)
USE_FULL_FT  = False       # True for full finetune (requires more VRAM); default is LoRA
SEED         = 3407

# Data source options (pick ONE path below)
USE_OSCAR       = True     # pull public corpus for a target language
TARGET_LANG     = "sw"     # ISO code (e.g., "sw" Swahili, "ta" Tamil, "id" Indonesian, "tr", "bn", "fa", etc.)
OSCAR_DATASET   = "oscar-corpus/OSCAR-2301"   # large; we will subset
OSCAR_SPLIT     = f"train[{TARGET_LANG}]"     # the HF dataset uses language configs; we’ll map below

# If you prefer a small curated dataset, set USE_OSCAR=False and USE_SAMPLE=True
USE_SAMPLE      = not USE_OSCAR

# Optional: upload your own TXT files instead (set USE_OSCAR=False and USE_SAMPLE=False)
USE_UPLOAD      = False  # If True, run the upload cell below

OUTPUT_DIR = f"continued_pt_{TARGET_LANG}_smollm2"
MERGED_DIR = f"{OUTPUT_DIR}_merged"


In [4]:
import torch
from unsloth import FastLanguageModel

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = dtype,
    load_in_4bit   = USE_4BIT,
)

# Ensure PAD token is set
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
try:
    model.config.pad_token_id = tokenizer.pad_token_id
except Exception:
    pass

if not USE_FULL_FT:
    # Attach LoRA adapters for low-VRAM continued pretraining
    model = FastLanguageModel.get_peft_model(
        model,
        r=16, lora_alpha=16, lora_dropout=0.0,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing=True,
        random_state=SEED,
        max_seq_length=MAX_SEQ_LEN,
    )

print("✅ Model ready. LoRA:", (not USE_FULL_FT))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.11.2 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


✅ Model ready. LoRA: True


In [8]:
# --- Public alternative: Wikimedia Wikipedia (streams, ungated) ---
# Uses language-specific snapshots like "20231101.sw"
from datasets import load_dataset, Dataset

TARGET_LANG = globals().get("TARGET_LANG", "sw")

# Try a few recent snapshots; we’ll use the first that works.
_snapshot_candidates = [
    f"20241001.{TARGET_LANG}",
    f"20240701.{TARGET_LANG}",
    f"20240401.{TARGET_LANG}",
    f"20240101.{TARGET_LANG}",
    f"20231101.{TARGET_LANG}",
]

raw_ds = None
last_err = None
for snap in _snapshot_candidates:
    try:
        # Wikipedia dump has {"id","url","title","text"}
        stream = load_dataset("wikimedia/wikipedia", snap, split="train", streaming=True)
        # Take a small shard for fast runs (adjust 30_000 higher for better quality)
        def take_n(s, n=30_000):
            for i, x in enumerate(s):
                if i >= n: break
                yield x
        texts = []
        for row in take_n(stream):
            t = (row.get("text") or "").strip()
            if t:
                texts.append(t)
        raw_ds = Dataset.from_dict({"text": texts})
        print(f"✅ Loaded Wikipedia snapshot: {snap} | samples: {len(raw_ds)}")
        break
    except Exception as e:
        last_err = e
        print(f"⚠️ Snapshot {snap} failed: {e}")

if raw_ds is None:
    raise SystemExit(
        f"❌ Could not load any wikipedia snapshot for lang='{TARGET_LANG}'. "
        f"Try a different TARGET_LANG or set USE_SAMPLE=True. Last error: {last_err}"
    )

print(raw_ds[0])


README.md: 0.00B [00:00, ?B/s]

⚠️ Snapshot 20241001.sw failed: BuilderConfig '20241001.sw' not found. Available: ['20231101.ab', '20231101.ace', '20231101.ady', '20231101.af', '20231101.als', '20231101.alt', '20231101.am', '20231101.ami', '20231101.an', '20231101.ang', '20231101.anp', '20231101.ar', '20231101.arc', '20231101.ary', '20231101.arz', '20231101.as', '20231101.ast', '20231101.atj', '20231101.av', '20231101.avk', '20231101.awa', '20231101.ay', '20231101.az', '20231101.azb', '20231101.ba', '20231101.ban', '20231101.bar', '20231101.bat-smg', '20231101.bcl', '20231101.be', '20231101.be-x-old', '20231101.bg', '20231101.bh', '20231101.bi', '20231101.bjn', '20231101.blk', '20231101.bm', '20231101.bn', '20231101.bo', '20231101.bpy', '20231101.br', '20231101.bs', '20231101.bug', '20231101.bxr', '20231101.ca', '20231101.cbk-zam', '20231101.cdo', '20231101.ce', '20231101.ceb', '20231101.ch', '20231101.chr', '20231101.chy', '20231101.ckb', '20231101.co', '20231101.cr', '20231101.crh', '20231101.cs', '20231101.csb', '